<a href="https://colab.research.google.com/github/glwat/Durham_Masters/blob/main/Whitby_Data_Download_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

These scripts were written by Gethin Watkins, with the help of Microsoft Copilot for downloading data from the Whitby Weather Underground station. These scripts are based on scripts written by Nick Rosser.

Download data from WUnderground to google colab file storage as individual csv files, one csv for each day.

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
import os

# ----------------------------------------
# CONFIG
# ----------------------------------------

API_KEY = "7e9adeec8fee4d309adeec8fee5d3088"
STATION_ID = "INORTHYO14"

URL1 = (
    "https://api.weather.com/v2/pws/history/hourly?"
    f"stationId={STATION_ID}&format=json&units=m&numericPrecision=decimal&date="
)
URL2 = f"&apiKey={API_KEY}"

start_date = datetime(2021, 1, 1)
end_date   = datetime(2022, 1, 1)

# Output directory (Colab default)
OUTPUT_DIR = "/content/wunderground_whitby/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------
# VARIABLES TO EXTRACT (matching Stata)
# ----------------------------------------

fields = [
    "obsTimeLocal", "epoch", "solarRadiationHigh", "uvHigh", "winddirAvg",
    "humidityHigh", "humidityLow", "humidityAvg",
    "metric.tempHigh", "metric.tempLow", "metric.tempAvg",
    "metric.windspeedHigh", "metric.windspeedLow", "metric.windspeedAvg",
    "metric.windgustHigh", "metric.windgustLow", "metric.windgustAvg",
    "metric.dewptHigh", "metric.dewptLow", "metric.dewptAvg",
    "metric.windchillHigh", "metric.windchillLow", "metric.windchillAvg",
    "metric.heatindexHigh", "metric.heatindexLow", "metric.heatindexAvg",
    "metric.pressureMax", "metric.pressureMin", "metric.pressureTrend",
    "metric.precipRate", "metric.precipTotal"
]

# ----------------------------------------
# MAIN LOOP
# ----------------------------------------

current = start_date
obs = 0  # equivalent to Stata offset()

while current <= end_date:
    date_str = current.strftime("%Y%m%d")  # CCYYNNDD format
    url = f"{URL1}{date_str}{URL2}"

    print(f"Fetching {date_str} ...")

    try:
        r = requests.get(url)
        r.raise_for_status()
        data = r.json()

        if "observations" not in data:
            print(f"⚠️ No observations for {date_str}")
            current += timedelta(days=1)
            continue

        # Flatten JSON
        df = pd.json_normalize(data["observations"])

        # Keep only the fields you want
        df = df.reindex(columns=fields)

        # Save CSV
        out_path = os.path.join(OUTPUT_DIR, f"{date_str}.csv")
        df.to_csv(out_path, index=False)

        print(f"Saved → {out_path}")

    except Exception as e:
        print(f"❌ Error on {date_str}: {e}")

    # Sleep 2000 ms (2 seconds)
    time.sleep(2)

    current += timedelta(days=1)
    obs += 1

Fetching 20210101 ...
Saved → /content/wunderground_whitby/20210101.csv
Fetching 20210102 ...
Saved → /content/wunderground_whitby/20210102.csv
Fetching 20210103 ...
Saved → /content/wunderground_whitby/20210103.csv
Fetching 20210104 ...
Saved → /content/wunderground_whitby/20210104.csv
Fetching 20210105 ...
Saved → /content/wunderground_whitby/20210105.csv
Fetching 20210106 ...
Saved → /content/wunderground_whitby/20210106.csv
Fetching 20210107 ...
Saved → /content/wunderground_whitby/20210107.csv
Fetching 20210108 ...
Saved → /content/wunderground_whitby/20210108.csv
Fetching 20210109 ...
Saved → /content/wunderground_whitby/20210109.csv
Fetching 20210110 ...
Saved → /content/wunderground_whitby/20210110.csv
Fetching 20210111 ...
Saved → /content/wunderground_whitby/20210111.csv
Fetching 20210112 ...
Saved → /content/wunderground_whitby/20210112.csv
Fetching 20210113 ...
Saved → /content/wunderground_whitby/20210113.csv
Fetching 20210114 ...
Saved → /content/wunderground_whitby/20210

Inspect data for a single day:

In [ ]:
import pandas as pd

df = pd.read_csv("/content/wunderground_whitby/20210101.csv")
df.head()

,obsTimeLocal,epoch,solarRadiationHigh,uvHigh,winddirAvg,humidityHigh,humidityLow,humidityAvg,metric.tempHigh,metric.tempLow,...,metric.windchillLow,metric.windchillAvg,metric.heatindexHigh,metric.heatindexLow,metric.heatindexAvg,metric.pressureMax,metric.pressureMin,metric.pressureTrend,metric.precipRate,metric.precipTotal
0,2021-01-01 00:45:33,1609461933,NaN,NaN,301,91.0,88.0,89.5,4.4,3.7,...,3.7,4.0,4.4,3.7,4.0,1007.69,1007.45,0.32,0.00,0.00
1,2021-01-01 01:45:33,1609465533,NaN,NaN,313,92.0,90.0,91.3,5.1,3.7,...,0.7,2.1,5.1,3.7,4.5,1008.06,1007.76,0.41,0.51,0.51
2,2021-01-01 02:45:33,1609469133,NaN,NaN,333,90.0,89.0,89.5,5.8,4.3,...,3.3,3.7,5.8,4.3,5.1,1008.06,1007.96,0.14,0.51,0.51
3,2021-01-01 03:45:33,1609472733,NaN,NaN,307,91.0,86.0,87.8,6.3,5.1,...,2.4,3.4,6.3,5.1,5.8,1008.47,1008.26,0.14,0.00,0.51
4,2021-01-01 04:45:33,1609476333,NaN,NaN,333,89.0,88.0,88.5,6.1,5.6,...,1.5,2.9,6.1,5.6,5.8,1008.47,1008.13,-0.27,0.76,1.27


Define a function to append all csv files (each containing a single day's observations) into a dataframe containing data across the entire of 2021:

In [ ]:
import pandas as pd
import os

def load_and_append_all_csvs(folder="/content/wunderground_whitby/"):
    """
    Loads all CSVs in a folder, checks that they share identical columns,
    and appends them into one combined DataFrame.
    """

    files = sorted([f for f in os.listdir(folder) if f.endswith(".csv")])

    if not files:
        raise ValueError("No CSV files found in the folder.")

    print(f"Found {len(files)} CSV files.")

    # Load first file to establish the reference schema
    first_df = pd.read_csv(os.path.join(folder, files[0]))
    reference_cols = list(first_df.columns)

    print("Reference column set established from:", files[0])
    print(reference_cols)

    dfs = [first_df]

    # Loop through remaining files
    for f in files[1:]:
        path = os.path.join(folder, f)
        df = pd.read_csv(path)

        # Check column consistency
        if list(df.columns) != reference_cols:
            print(f"⚠️ Column mismatch in file: {f}")
            print("Expected:", reference_cols)
            print("Found:   ", list(df.columns))
            raise ValueError("Column mismatch detected — stopping to avoid corrupt append.")

        dfs.append(df)

    # Append all
    combined = pd.concat(dfs, ignore_index=True)
    print(f"Combined dataset contains {combined.shape[0]} rows and {combined.shape[1]} columns.")

    return combined

Execute the command defined above and inspect the variable columns:

In [ ]:
combined_df = load_and_append_all_csvs()
combined_df.info()

Found 367 CSV files.
Reference column set established from: 20170101.csv
['obsTimeLocal', 'epoch', 'solarRadiationHigh', 'uvHigh', 'winddirAvg', 'humidityHigh', 'humidityLow', 'humidityAvg', 'metric.tempHigh', 'metric.tempLow', 'metric.tempAvg', 'metric.windspeedHigh', 'metric.windspeedLow', 'metric.windspeedAvg', 'metric.windgustHigh', 'metric.windgustLow', 'metric.windgustAvg', 'metric.dewptHigh', 'metric.dewptLow', 'metric.dewptAvg', 'metric.windchillHigh', 'metric.windchillLow', 'metric.windchillAvg', 'metric.heatindexHigh', 'metric.heatindexLow', 'metric.heatindexAvg', 'metric.pressureMax', 'metric.pressureMin', 'metric.pressureTrend', 'metric.precipRate', 'metric.precipTotal']
Combined dataset contains 8365 rows and 31 columns.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8365 entries, 0 to 8364
Data columns (total 31 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   obsTimeLocal          8365 non-null   obje

/tmp/ipykernel_291/3373160137.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(dfs, ignore_index=True)


Export the new dataframe to a file in the google drive:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

output_path = "/content/drive/My Drive/Masters_Colab_Data/Whitby_WUnderground.xlsx"

combined_df.to_excel(output_path, index=False)

print("Saved to:", output_path)

Mounted at /content/drive
Saved to: /content/drive/My Drive/Masters_Colab_Data/Whitby_WUnderground.xlsx
